In [0]:
import os, sys
sys.path.append(os.path.abspath('..'))

from src import regras as R
from src import transformacoes as T
from src.config import CATALOGO
from src.silver import merge_clientes, processar_fluxo

dbutils.widgets.text('fluxos', 'todos')
filtro = [f.strip() for f in dbutils.widgets.get('fluxos').split(',')]
print(f'Fluxos solicitados: {filtro}')

B, S = f'{CATALOGO}.bronze', f'{CATALOGO}.silver'

FLUXOS = {
    'cotacoes_awesomeapi': dict(
        tabela_bronze=f'{B}.awesomeapi_cotacoes_atuais', tabela_silver=f'{S}.cotacoes_intradiarias',
        transformar=T.cotacoes_awesome, regras=R.COTACOES,
        chaves=['ativo', 'fonte', 'cotado_em_utc'], ordem=['coletado_em_utc']),
    'cotacoes_brapi': dict(
        tabela_bronze=f'{B}.brapi_cotacoes', tabela_silver=f'{S}.cotacoes_intradiarias',
        transformar=T.cotacoes_brapi, regras=R.COTACOES,
        chaves=['ativo', 'fonte', 'cotado_em_utc'], ordem=['coletado_em_utc']),
    'precos_awesomeapi': dict(
        tabela_bronze=f'{B}.awesomeapi_historico_diario', tabela_silver=f'{S}.precos_diarios',
        transformar=T.precos_awesome, regras=R.PRECOS_DIARIOS,
        chaves=['ativo', 'fonte', 'data_pregao'], ordem=['coletado_em_utc', 'cotado_em_utc']),
    'precos_brapi': dict(
        tabela_bronze=f'{B}.brapi_historico', tabela_silver=f'{S}.precos_diarios',
        transformar=T.precos_brapi, regras=R.PRECOS_DIARIOS,
        chaves=['ativo', 'fonte', 'data_pregao'], ordem=['coletado_em_utc', 'cotado_em_utc']),
    'ativos_awesomeapi': dict(
        tabela_bronze=f'{B}.awesomeapi_catalogo_pares', tabela_silver=f'{S}.ativos',
        transformar=T.ativos_awesome, regras=R.ATIVOS,
        chaves=['ativo'], ordem=['coletado_em_utc']),
    'ativos_brapi': dict(
        tabela_bronze=f'{B}.brapi_catalogo_ativos', tabela_silver=f'{S}.ativos',
        transformar=T.ativos_brapi, regras=R.ATIVOS,
        chaves=['ativo'], ordem=['coletado_em_utc']),
    'indicadores_bcb': dict(
        tabela_bronze=f'{B}.bcb_series', tabela_silver=f'{S}.indicadores_economicos',
        transformar=lambda df: T.indicadores_bcb(df, spark.table(f'{S}.ref_series_bcb')),
        regras=R.INDICADORES,
        chaves=['codigo_sgs', 'data_referencia'], ordem=['coletado_em_utc']),
    'tesouro': dict(
        tabela_bronze=f'{B}.tesouro_precos_taxas', tabela_silver=f'{S}.tesouro_precos_taxas',
        transformar=T.tesouro, regras=R.TESOURO,
        chaves=['tipo_titulo', 'data_vencimento', 'data_base'], ordem=['_ingerido_em', '_arquivo_origem']),
    'clientes': dict(
        tabela_bronze=f'{B}.clientes', tabela_silver=f'{S}.clientes',
        transformar=T.clientes_eventos, regras=R.CLIENTES,
        chaves=['id_cliente'], ordem=['ultimo_evento_em_utc'],
        consolidar=T.consolidar_clientes, merge=merge_clientes),
    'operacoes': dict(
        tabela_bronze=f'{B}.operacoes', tabela_silver=f'{S}.operacoes',
        transformar=T.operacoes, regras=R.OPERACOES,
        chaves=['id_operacao'], ordem=['_ingerido_em', '_arquivo_origem']),
}

erros = []
for fluxo, parametros in FLUXOS.items():
    if 'todos' not in filtro and fluxo not in filtro:
        continue
    if not spark.catalog.tableExists(parametros['tabela_bronze']):
        print(f'[pulado] {fluxo}: tabela Bronze ainda não existe')
        continue
    try:
        processar_fluxo(fluxo, **parametros)
    except Exception as e:
        erros.append(f'{fluxo}: {type(e).__name__}: {str(e)[:300]}')
        print(f'[erro] {fluxo}')

if erros:
    raise RuntimeError('Falhas na Silver -> ' + ' | '.join(erros))